<a href="https://colab.research.google.com/github/Ehtel-crypto/Random-omok/blob/main/random-omok.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%html
<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>랜덤 돌 오목</title>
<style>
:root{--ink:#212a35;--muted:#697586;--accent:#176b58;--gold:#e9bd69;--paper:#fffdf7;--board:#e8bd79;--line:#593d25}
*{box-sizing:border-box}body{margin:0;background:linear-gradient(145deg,#f4f6f1,#e8eee9);font-family:system-ui,-apple-system,"Segoe UI",sans-serif;color:var(--ink);min-height:100vh;padding:24px}
.app{max-width:1100px;margin:auto}.top{display:flex;align-items:end;justify-content:space-between;gap:20px;margin-bottom:18px}.eyebrow{font-size:12px;letter-spacing:.15em;color:var(--accent);font-weight:800}.title{font-size:clamp(28px,5vw,42px);margin:4px 0 0;letter-spacing:-.04em}.subtitle{color:var(--muted);margin:7px 0 0;font-size:14px}.layout{display:grid;grid-template-columns:minmax(0,1fr) 290px;gap:20px;align-items:start}.board-card,.panel{background:var(--paper);border:1px solid #e2e7df;border-radius:18px;box-shadow:0 12px 34px #19372b12}.board-card{padding:clamp(10px,2.5vw,24px)}.board-wrap{width:min(100%,720px);margin:auto;aspect-ratio:1;background:var(--board);border-radius:8px;box-shadow:inset 0 2px 9px #744c2828;overflow:hidden}canvas{display:block;width:100%;height:100%;touch-action:manipulation}.panel{padding:20px;position:sticky;top:20px}.panel h2{font-size:17px;margin:0 0 14px}.players{display:grid;grid-template-columns:1fr 1fr;gap:10px}.player{padding:12px;border-radius:13px;background:#f3f5f1;border:2px solid transparent}.player.active{border-color:var(--accent);background:#edf6f1}.player-top{display:flex;align-items:center;gap:8px;font-weight:800}.stone-icon{width:17px;height:17px;border-radius:50%;display:inline-block;box-shadow:inset -2px -3px 4px #0003}.black{background:#202124}.white{background:#fff;border:1px solid #bbb}.count{font-size:25px;font-weight:850;margin:9px 0 0}.count small{font-size:12px;font-weight:600;color:var(--muted)}.turn-box{margin:16px 0 10px;padding:13px;border-radius:12px;background:#f5f7f4}.turn-label{font-size:12px;color:var(--muted)}#status{font-weight:750;margin-top:5px;line-height:1.45}.buttons{display:flex;gap:9px;margin-top:14px}.btn{border:0;border-radius:10px;padding:11px 14px;font-size:14px;font-weight:750;cursor:pointer}.primary{background:var(--accent);color:white;flex:1}.secondary{background:#edf0ec;color:#36433e}.btn:hover{filter:brightness(.96)}.rules{margin-top:20px;border-top:1px solid #e8ebe6;padding-top:15px;color:var(--muted);font-size:13px;line-height:1.65}.rules strong{color:#36433e}.legend{display:flex;gap:14px;flex-wrap:wrap;font-size:12px;color:var(--muted);margin-top:12px}.legend span{display:flex;align-items:center;gap:6px}.block-icon{width:15px;height:15px;background:#52645c;border-radius:4px;display:inline-block}
@media(max-width:800px){body{padding:14px}.layout{grid-template-columns:1fr}.panel{position:static}.top{align-items:start;flex-direction:column}.board-card{padding:10px}.board-wrap{width:100%}}
</style>
</head>
<body><main class="app">
<header class="top"><div><div class="eyebrow">RANDOM STONE OMOK</div><h1 class="title">랜덤 돌 오목</h1><p class="subtitle">매 턴 무작위로 지급되는 돌을 활용해 먼저 5개를 이어보세요.</p></div></header>
<section class="layout"><div class="board-card"><div class="board-wrap"><canvas id="board" aria-label="19 곱하기 19 오목판"></canvas></div><div class="legend"><span><i class="stone-icon black"></i> 흑돌</span><span><i class="stone-icon white"></i> 백돌</span><span><i class="block-icon"></i> 장애물</span></div></div>
<aside class="panel"><h2>게임 현황</h2><div class="players"><div class="player" id="blackCard"><div class="player-top"><i class="stone-icon black"></i> 흑</div><div class="count" id="blackCount">— <small>개</small></div></div><div class="player" id="whiteCard"><div class="player-top"><i class="stone-icon white"></i> 백</div><div class="count" id="whiteCount">— <small>개</small></div></div></div><div class="turn-box"><div class="turn-label" id="round">게임 시작 전</div><div id="status">새 게임을 시작해 주세요.</div></div><div class="buttons"><button class="btn primary" id="newGame">새 게임</button><button class="btn secondary" id="undo">한 수 무르기</button></div><div class="rules"><strong>진행 방법</strong><br>매 라운드 흑·백에게 각각 1~3개의 돌을 무작위 지급합니다. 돌이 더 많은 쪽이 먼저 두며, 이후에는 번갈아 둡니다. 한쪽의 돌이 모두 소진되면 남은 돌이 있는 쪽이 계속 둡니다.<br><br><strong>목표</strong><br>가로·세로·대각선 중 한 방향으로 같은 색 돌 5개를 연속으로 놓으면 승리합니다. 장애물은 돌을 놓을 수 없으며, 새 게임마다 무작위 배치됩니다.</div></aside></section></main>
<script>
((()=>{
const N=19, canvas=document.getElementById('board'),ctx=canvas.getContext('2d');
let cells=[],stones=[],counts={1:0,2:0},roundNo=0,active=0,roundStarter=0,gameOver=false,history=[];
const status=document.getElementById('status'),roundEl=document.getElementById('round');
const dirs=[[1,0],[0,1],[1,1],[1,-1]];
function resize(){const dpr=window.devicePixelRatio||1,rect=canvas.getBoundingClientRect();canvas.width=Math.round(rect.width*dpr);canvas.height=Math.round(rect.height*dpr);ctx.setTransform(dpr,0,0,dpr,0,0);draw()}
function draw(){const w=canvas.clientWidth,h=canvas.clientHeight;if(!w||!h)return;ctx.clearRect(0,0,w,h);const pad=Math.max(13,w*.035),step=(w-2*pad)/(N-1),r=Math.max(4,step*.39);
ctx.strokeStyle='#694a2b';ctx.lineWidth=1;for(let i=0;i<N;i++){let p=pad+i*step;ctx.beginPath();ctx.moveTo(pad,p);ctx.lineTo(w-pad,p);ctx.stroke();ctx.beginPath();ctx.moveTo(p,pad);ctx.lineTo(p,h-pad);ctx.stroke()}
[3,9,15].forEach(x=>[3,9,15].forEach(y=>{ctx.fillStyle='#694a2b';ctx.beginPath();ctx.arc(pad+x*step,pad+y*step,Math.max(2,step*.09),0,Math.PI*2);ctx.fill()}));
for(let y=0;y<N;y++)for(let x=0;x<N;x++){let px=pad+x*step,py=pad+y*step;if(cells[y][x]===3){ctx.fillStyle='#52645c';ctx.beginPath();ctx.roundRect(px-r*.77,py-r*.77,r*1.54,r*1.54,Math.max(2,r*.2));ctx.fill();ctx.strokeStyle='#d7e0d8';ctx.lineWidth=1.3;ctx.beginPath();ctx.moveTo(px-r*.42,py-r*.42);ctx.lineTo(px+r*.42,py+r*.42);ctx.moveTo(px+r*.42,py-r*.42);ctx.lineTo(px-r*.42,py+r*.42);ctx.stroke()}else if(cells[y][x]===1||cells[y][x]===2){ctx.beginPath();ctx.arc(px,py,r,0,Math.PI*2);ctx.fillStyle=cells[y][x]===1?'#202124':'#fff';ctx.fill();ctx.strokeStyle=cells[y][x]===1?'#090a0b':'#909090';ctx.lineWidth=1;ctx.stroke();if(stones.length&&stones[stones.length-1].x===x&&stones[stones.length-1].y===y){ctx.fillStyle=cells[y][x]===1?'#f6d27e':'#b6382c';ctx.beginPath();ctx.arc(px,py,Math.max(1.7,r*.17),0,Math.PI*2);ctx.fill()}}}}
function update(){document.getElementById('blackCount').innerHTML=`${counts[1]} <small>개</small>`;document.getElementById('whiteCount').innerHTML=`${counts[2]} <small>개</small>`;document.getElementById('blackCard').classList.toggle('active',!gameOver&&active===1);document.getElementById('whiteCard').classList.toggle('active',!gameOver&&active===2);draw()}
function start(){cells=Array.from({length:N},()=>Array(N).fill(0));stones=[];history=[];counts={1:0,2:0};roundNo=0;gameOver=false;let placed=0;while(placed<22){let x=Math.floor(Math.random()*N),y=Math.floor(Math.random()*N);if(cells[y][x]===0){cells[y][x]=3;placed++}};nextRound();update()}
function nextRound(){if(gameOver)return;roundNo++;counts[1]=1+Math.floor(Math.random()*3);counts[2]=1+Math.floor(Math.random()*3);roundStarter=counts[1]===counts[2]?(Math.random()<.5?1:2):(counts[1]>counts[2]?1:2);
active=roundStarter;roundEl.textContent=`${roundNo}라운드 · 흑 ${counts[1]}개 / 백 ${counts[2]}개`;status.textContent=`${active===1?'흑':'백'}이 먼저 둡니다.`;update()}
function findWin(x,y,color){for(const [dx,dy]of dirs){let line=[[x,y]];for(const s of [-1,1]){let nx=x+s*dx,ny=y+s*dy;while(nx>=0&&ny>=0&&nx<N&&ny<N&&cells[ny][nx]===color){line.push([nx,ny]);nx+=s*dx;ny+=s*dy}}if(line.length>=5)return line}return null}
function click(e){if(gameOver||!roundNo)return;let rect=canvas.getBoundingClientRect(),pad=Math.max(13,rect.width*.035),step=(rect.width-2*pad)/(N-1),x=Math.round((e.clientX-rect.left-pad)/step),y=Math.round((e.clientY-rect.top-pad)/step);if(x<0||y<0||x>=N||y>=N||cells[y][x]!==0)return;
history.push({x,y,color:active,counts:{...counts},active,roundNo,roundStarter});cells[y][x]=active;counts[active]--;let win=findWin(x,y,active);if(win){gameOver=true;status.textContent=`${active===1?'흑':'백'} 승리! 5목을 완성했습니다.`;update();return}let other=active===1?2:1;if(counts[other]>0)active=other;else if(counts[active]>0){}else{nextRound();return}status.textContent=`${active===1?'흑':'백'} 차례입니다.`;update()}
function undo(){if(!history.length||gameOver)return;let prev=history.pop();cells[prev.y][prev.x]=0;counts=prev.counts;active=prev.active;roundNo=prev.roundNo;roundStarter=prev.roundStarter;roundEl.textContent=`${roundNo}라운드 · 흑 ${counts[1]}개 / 백 ${counts[2]}개`;status.textContent=`${active===1?'흑':'백'} 차례입니다.`;update()}
canvas.addEventListener('click',click);document.getElementById('newGame').addEventListener('click',start);document.getElementById('undo').addEventListener('click',undo);window.addEventListener('resize',resize);resize();start();
}))();
</script></body></html>